# 🌸 HaruDrive Ultra-Speed Batch Cloud Mirror
Notebook ini memindahkan file/folder dari Google Drive ke Hugging Face Dataset Storage secara langsung (**Multi-Threaded 1-Commit-per-Batch**).

### 🔑 Kredensial Secrets Colab (Ikon Kunci 🔑 di sidebar kiri):
- `GDRIVE_CLIENT_ID`: OAuth Client ID Google Drive
- `GDRIVE_CLIENT_SECRET`: OAuth Client Secret Google Drive
- `GDRIVE_REFRESH_TOKEN`: OAuth Refresh Token Google Drive
- `HF_TOKEN`: Hugging Face Access Token (Write Permission)
- `HF_REPO_ID`: Nama Hugging Face Dataset Anda (misal: `username/nama-dataset`)

In [ ]:
# @title 🌸 HaruDrive Ultra-Speed Batch Cloud Mirror (Anti-Rate-Limit & Anti-Gagal)
# @markdown Memproses file per batch (10-15 file sekaligus) dan mengunggahnya dalam **1 Commit per Batch** (Bebas dari limit 128 commit/jam HF).

import os
import sys
import time
import json
import shutil
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Form Input (Opsional jika sudah diset di Colab Secrets)
GDRIVE_URL = "" # @param {type:"string"}
TARGET_HF_PATH = "" # @param {type:"string"}
HF_REPO_ID = "" # @param {type:"string"}
FOLDER_NAME = "" # @param {type:"string"}
BATCH_SIZE = 12 # @param {type:"slider", min:5, max:25, step:1}

# 2. Ambil Kredensial dari Colab Secrets (Ikon Kunci 🔑 di Sidebar)
from google.colab import userdata

try:
    client_id = userdata.get('GDRIVE_CLIENT_ID')
    client_secret = userdata.get('GDRIVE_CLIENT_SECRET')
    refresh_token = userdata.get('GDRIVE_REFRESH_TOKEN')
    HF_TOKEN = userdata.get('HF_TOKEN')
    secret_repo_id = userdata.get('HF_REPO_ID')
    if secret_repo_id:
        HF_REPO_ID = secret_repo_id
except Exception as e:
    print(f"❌ Gagal membaca Secrets: {e}")
    print("👉 Pastikan Anda sudah menambahkan GDRIVE_CLIENT_ID, GDRIVE_CLIENT_SECRET, GDRIVE_REFRESH_TOKEN, HF_TOKEN, dan HF_REPO_ID di menu Secrets (🔑) Colab dan menyalakan saklar 'Notebook access'!")
    sys.exit(1)

if not HF_TOKEN:
    print("❌ Secret 'HF_TOKEN' belum diset di ikon 🔑 Secrets Colab!")
    sys.exit(1)

if not HF_REPO_ID:
    print("❌ Masukkan HF_REPO_ID (misal: username/nama-dataset) pada form atau pada Colab Secrets (🔑)!")
    sys.exit(1)

# 3. Setup Dependencies & Autentikasi
print("⚡ Mempersiapkan dependensi...")
try:
    import requests
    from huggingface_hub import HfApi, login
except ImportError:
    !pip install -q huggingface_hub requests
    import requests
    from huggingface_hub import HfApi, login

print(f"🔑 Mengautentikasi ke Hugging Face Storage ({HF_REPO_ID})...")
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
try:
    api.repo_info(repo_id=HF_REPO_ID, repo_type="dataset")
    print(f"✅ Terhubung ke Hugging Face Dataset: {HF_REPO_ID}")
except Exception:
    print(f"📦 Membuat dataset '{HF_REPO_ID}' (Public)...")
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", private=False, exist_ok=True)

# 4b. Gofile helpers (filmbeehub proxy API, bypass notPremium limit)
GOFILE_API_BASE = "https://go.filmbeehub.workers.dev"
try:
    GOFILE_API_TOKEN = userdata.get('GOFILE_API_TOKEN')
except Exception:
    GOFILE_API_TOKEN = ""

def detect_source_kind(url):
    u = (url or "").strip().lower()
    return "gofile" if "gofile.io" in u else "gdrive"

def extract_gofile_id(url):
    u = (url or "").strip()
    if "/d/" in u:
        return u.split("/d/", 1)[1].split("?")[0].split("/")[0].split("#")[0]
    return u

def normalize_gofile_file(f):
    if not isinstance(f, dict):
        return None
    name = f.get("name") or f.get("fileName") or f.get("filename") or ""
    link = f.get("downloadUrl") or f.get("link") or f.get("url") or f.get("directLink") or f.get("content") or ""
    size = f.get("bytes") or f.get("size") or f.get("fileSize") or 0
    try:
        size = int(size)
    except Exception:
        try:
            import re as _re
            m = _re.match(r"\s*([\d.]+)\s*([KMGT]?B)", str(size), _re.I)
            mult = {"B": 1, "KB": 1024, "MB": 1024**2, "GB": 1024**3, "TB": 1024**4}
            size = int(float(m.group(1)) * mult.get(m.group(2).upper(), 1)) if m else 0
        except Exception:
            size = 0
    if not name:
        try:
            from urllib.parse import unquote as _unq
            name = _unq((link or "").rstrip("/").split("/")[-1].split("?")[0]) or "file"
        except Exception:
            name = "file"
    if not link:
        return None
    return {"name": name, "link": link, "size": size}

def parse_gofile_response(data):
    files, folder_name = [], ""
    if isinstance(data, dict):
        d = data.get("data")
        if isinstance(d, dict):
            dl = d.get("downloadLinks")
            if isinstance(dl, list):
                folder_name = d.get("name") or ""
                for f in dl:
                    n = normalize_gofile_file(f)
                    if n:
                        files.append(n)
                return files, folder_name
            ch = d.get("children")
            if isinstance(ch, dict):
                folder_name = d.get("name") or ""
                for _, f in ch.items():
                    n = normalize_gofile_file(f)
                    if n:
                        files.append(n)
                return files, folder_name
            if isinstance(ch, list):
                folder_name = d.get("name") or ""
                for f in ch:
                    n = normalize_gofile_file(f)
                    if n:
                        files.append(n)
                return files, folder_name
        for key in ("files", "links", "data"):
            c = data.get(key)
            if isinstance(c, list):
                for f in c:
                    n = normalize_gofile_file(f)
                    if n:
                        files.append(n)
                if files:
                    break
        for key in ("folderName", "folder_name", "name"):
            v = data.get(key)
            if isinstance(v, str) and v:
                folder_name = v
                break
    elif isinstance(data, list):
        for f in data:
            n = normalize_gofile_file(f)
            if n:
                files.append(n)
    return files, folder_name

def resolve_gofile_files(api_base, api_token, gofile_url, password="", page_size=100, max_pages=50):
    gid = extract_gofile_id(gofile_url)
    if not gid:
        raise Exception("Gofile ID tidak valid.")
    if not api_token:
        raise Exception("Secret 'GOFILE_API_TOKEN' belum diset di ikon Secrets Colab!")
    headers = {"Authorization": f"Bearer {api_token}", "Content-Type": "application/json"}
    files, folder_name, seen, page = [], "", set(), 0
    while page < max_pages:
        payload = {"url": f"https://gofile.io/d/{gid}", "password": password or "", "expiresInSeconds": 3600, "filePage": page, "filePageSize": page_size}
        data = None
        for api_attempt in range(1, 4):
            try:
                r = requests.post(api_base.rstrip("/") + "/api/v1/generate", json=payload, headers=headers, timeout=60)
                if r.status_code in (403, 429, 503):
                    if api_attempt < 3:
                        wait = 60 * api_attempt
                        print(f"    Gofile API throttled (HTTP {r.status_code}), retry {api_attempt}/3 in {wait}s...")
                        time.sleep(wait)
                        continue
                    raise Exception(f"Gofile API: HTTP {r.status_code} - upstream throttled, coba lagi nanti.")
                data = r.json()
                break
            except Exception as e:
                if api_attempt >= 3:
                    raise Exception(f"Gofile API error (page {page}): {e}")
                time.sleep(10)
        if data is None:
            raise Exception(f"Gofile API error (page {page}): no response after retries.")
        if isinstance(data, dict) and data.get("ok") is False and "status" not in data:
            raise Exception(f"Gofile API: {data.get('error', 'unknown error')}")
        batch, bname = parse_gofile_response(data)
        if bname and not folder_name:
            folder_name = bname
        new_count = 0
        for f in batch:
            key = f["link"] or f["name"]
            if key in seen:
                continue
            seen.add(key)
            files.append(f)
            new_count += 1
        try:
            dd = data.get("data", {}) if isinstance(data, dict) else {}
            has_more = dd.get("hasMoreFiles", None)
        except Exception:
            has_more = None
        if has_more is False:
            break
        if has_more is True:
            page += 1
            continue
        if len(batch) < page_size or new_count == 0:
            break
        page += 1
    return files, folder_name, gid

def download_url_stream(url, dest_p, max_retries=4):
    import time as _time2
    os.makedirs(os.path.dirname(dest_p), exist_ok=True)
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, headers={"User-Agent": "HaruDrive-Mirror/1.0"}, stream=True, timeout=180) as r:
                if r.status_code in (429, 503):
                    if attempt < max_retries:
                        wait = 30 * attempt
                        print(f"    Rate limited upstream (HTTP {r.status_code}), retry {attempt}/{max_retries} in {wait}s...")
                        _time2.sleep(wait)
                        continue
                r.raise_for_status()
                with open(dest_p, "wb") as f:
                    for chunk in r.iter_content(chunk_size=16*1024*1024):
                        if chunk:
                            f.write(chunk)
            return dest_p
        except Exception as e:
            last_err = e
            raise
    raise last_err

def decide_container(custom_name, api_folder_name, fallback_id, file_count):
    custom = (custom_name or "").strip("/\\ ")
    if custom:
        return custom
    if file_count <= 1:
        return ""
    if api_folder_name:
        return api_folder_name.strip("/\\ ")
    return f"Gofile_{fallback_id[:8]}"


# 4. Helper Functions
def extract_gdrive_id(u):
    if not u: return "", False
    u = u.strip()
    is_f = "folders/" in u or "drive/folders" in u
    if "/folders/" in u: return u.split("/folders/")[1].split("?")[0].split("/")[0], True
    if "/d/" in u: return u.split("/d/")[1].split("?")[0].split("/")[0], False
    if "id=" in u: return u.split("id=")[1].split("&")[0], is_f
    if "file/d/" in u: return u.split("file/d/")[1].split("/")[0], False
    return u, is_f

def get_access_token(cid, csec, rtoken):
    token_url = "https://oauth2.googleapis.com/token"
    payload = {"client_id": cid, "client_secret": csec, "refresh_token": rtoken, "grant_type": "refresh_token"}
    r = requests.post(token_url, data=payload, timeout=30)
    if r.status_code == 200:
        return r.json().get("access_token")
    else:
        raise Exception(f"OAuth Error ({r.status_code}): {r.text}")

def list_folder_api(fid, token, cur_path=""):
    headers = {"Authorization": f"Bearer {token}"}
    files = []
    page_token = None
    while True:
        params = {
            "q": f"'{fid}' in parents and trashed = false",
            "fields": "nextPageToken, files(id, name, mimeType, size)",
            "pageSize": 1000,
            "supportsAllDrives": "true",
            "includeItemsFromAllDrives": "true"
        }
        if page_token: params["pageToken"] = page_token
        res = requests.get("https://www.googleapis.com/drive/v3/files", headers=headers, params=params, timeout=30)
        res.raise_for_status()
        data = res.json()
        for item in data.get("files", []):
            iname = item.get("name", "untitled")
            if item.get("mimeType") == "application/vnd.google-apps.folder":
                sub_p = f"{cur_path}/{iname}".strip("/") if cur_path else iname
                files.extend(list_folder_api(item["id"], token, sub_p))
            else:
                rel_p = f"{cur_path}/{iname}".strip("/") if cur_path else iname
                files.append({"id": item["id"], "name": iname, "rel_path": rel_p, "size": int(item.get("size", 0))})
        page_token = data.get("nextPageToken")
        if not page_token: break
    return files

def download_file_stream(fid, token, dest_p):
    headers = {"Authorization": f"Bearer {token}"}
    url = f"https://www.googleapis.com/drive/v3/files/{fid}?alt=media&supportsAllDrives=true"
    os.makedirs(os.path.dirname(dest_p), exist_ok=True)
    with requests.get(url, headers=headers, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(dest_p, "wb") as f:
            for chunk in r.iter_content(chunk_size=16*1024*1024):
                if chunk: f.write(chunk)
    return dest_p

def upload_folder_batch_safe(folder_path, path_in_repo, max_retries=4):
    for attempt in range(1, max_retries + 1):
        try:
            api.upload_folder(
                folder_path=folder_path,
                path_in_repo=path_in_repo,
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                commit_message=f"Batch Upload: /{path_in_repo}"
            )
            return True
        except Exception as err:
            print(f"   ⚠️ [Retry {attempt}/{max_retries}] Gagal upload batch: {err}")
            time.sleep(10 * attempt)
    return False

# 5. Eksekusi Mirror secara Batch
gdrive_id, is_folder_url = extract_gdrive_id(GDRIVE_URL)
target_dir = TARGET_HF_PATH.strip("/\\ ")
source_kind = detect_source_kind(GDRIVE_URL)
custom_folder = (FOLDER_NAME or "").strip("/\\ ")

if source_kind == "gdrive" and not gdrive_id:
    print("❌ Masukkan GDRIVE_URL terlebih dahulu!")
    sys.exit(1)

if source_kind == "gofile":
    print(f"Resolving Gofile: {GDRIVE_URL}")
    try:
        go_files, go_folder_name, go_gid = resolve_gofile_files(GOFILE_API_BASE, GOFILE_API_TOKEN, GDRIVE_URL)
    except Exception as e:
        print(f"Error: {e}")
        sys.exit(1)
    print(f"Discovered {len(go_files)} files" + (f" in '{go_folder_name}'" if go_folder_name else "") + ".")
    go_ok, go_fail, go_skip = 0, 0, 0
    go_start = time.time()
    if not go_files:
        print("Tidak ada file yang bisa di-mirror.")
    else:
        container = decide_container(custom_folder, go_folder_name, go_gid, len(go_files))
        # skip files already on HF
        try:
            existing_hf = set(api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset"))
        except Exception:
            existing_hf = set()
        pending = []
        for gf in go_files:
            dest_guess = "/".join([p for p in [target_dir, container, gf["name"]] if p])
            if dest_guess in existing_hf:
                go_skip += 1
            else:
                pending.append(gf)
        print(f"Total: {len(go_files)} file | Sudah Ada (Skip): {go_skip} | Perlu di-mirror: {len(pending)} file")
        if not pending:
            print("Semua file sudah ada di Hugging Face!")
        elif len(pending) == 1 and not container and not target_dir:
            gf = pending[0]
            temp_dir = tempfile.mkdtemp(prefix="single_go_")
            local_f = os.path.join(temp_dir, gf["name"])
            try:
                print(f"Mengunduh {gf['name']}...")
                download_url_stream(gf["link"], local_f)
                print(f"Mengunggah /{gf['name']}...")
                api.upload_file(path_or_fileobj=local_f, path_in_repo=gf["name"], repo_id=HF_REPO_ID, repo_type="dataset", commit_message=f"Mirror: {gf['name']}")
                print(f"Sukses: /{gf['name']}")
                go_ok = 1
            except Exception as e:
                print(f"Gagal: {e}")
                go_fail = 1
            finally:
                shutil.rmtree(temp_dir, ignore_errors=True)
        else:
            batches = [pending[j:j + BATCH_SIZE] for j in range(0, len(pending), BATCH_SIZE)]
            total_batches = len(batches)
            print(f"Memproses {len(pending)} file dalam {total_batches} Batch (1 Batch = 1 Commit)...")
            for b_idx, batch in enumerate(batches, 1):
                print("=" * 60)
                print(f"Memproses BATCH [{b_idx}/{total_batches}] ({len(batch)} file)...")
                temp_batch_dir = tempfile.mkdtemp(prefix=f"batch_{b_idx}_")
                batch_staging = os.path.join(temp_batch_dir, "staging")
                os.makedirs(batch_staging, exist_ok=True)
                print(f"Mengunduh {len(batch)} file secara paralel...")
                with ThreadPoolExecutor(max_workers=min(len(batch), 10)) as executor:
                    future_to_item = {}
                    for item in batch:
                        local_f = os.path.join(batch_staging, item["name"])
                        fut = executor.submit(download_url_stream, item["link"], local_f)
                        future_to_item[fut] = item
                    for fut in as_completed(future_to_item):
                        item = future_to_item[fut]
                        try:
                            fut.result()
                            sz_mb = item["size"] / (1024*1024)
                            print(f"   Selesai download: {item['name']} ({sz_mb:.1f} MB)")
                        except Exception as e:
                            print(f"   Gagal download {item['name']}: {e}")
                            go_fail += 1
                final_prefix_go = "/".join([p for p in [target_dir, container] if p])
                print(f"Mengunggah kloter batch ke Hugging Face: /{final_prefix_go} (1 Commit)...")
                try:
                    api.upload_folder(folder_path=batch_staging, path_in_repo=final_prefix_go, repo_id=HF_REPO_ID, repo_type="dataset", commit_message=f"Batch Upload: /{final_prefix_go}")
                    print(f"   BATCH [{b_idx}/{total_batches}] BERHASIL TERUPLOAD (1 Commit)!")
                    go_ok += len(batch)
                except Exception as e:
                    print(f"   BATCH [{b_idx}/{total_batches}] GAGAL UPLOAD: {e}")
                    go_fail += len(batch)
                shutil.rmtree(temp_batch_dir, ignore_errors=True)
                print(f"Disk kloter [{b_idx}/{total_batches}] telah dibersihkan.")
                time.sleep(2)
        go_dur = time.time() - go_start
        print("=" * 60)
        print(f"SELESAI dalam {go_dur/60:.2f} menit ({go_dur:.1f} detik)!")
        print(f"Berhasil: {go_ok} file | Sudah Ada (Skip): {go_skip} file | Gagal: {go_fail}")
else:
    access_token = get_access_token(client_id, client_secret, refresh_token)
    headers = {"Authorization": f"Bearer {access_token}"}
    m_res = requests.get(f"https://www.googleapis.com/drive/v3/files/{gdrive_id}?fields=id,name,mimeType,size&supportsAllDrives=true", headers=headers)
    meta = m_res.json() if m_res.status_code == 200 else {}
    is_folder = is_folder_url or (meta.get("mimeType") == "application/vnd.google-apps.folder")

    # Ambil daftar file yang SUDAH ada di HF agar tidak download ulang
    print(f"🔍 Memeriksa file yang sudah ada di Hugging Face ({HF_REPO_ID})...")
    existing_hf_files = set()
    try:
        hf_tree = api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset")
        existing_hf_files = set(hf_tree)
        print(f"📊 Terdeteksi {len(existing_hf_files)} file sudah ada di storage.")
    except Exception:
        pass

    start_time = time.time()
    ok_count = 0
    skip_count = 0
    fail_count = 0

    if is_folder:
        folder_name = meta.get("name", "Folder")
        print(f"📁 Memindai isi folder '{folder_name}'...")
        all_files = list_folder_api(gdrive_id, access_token)
        total_files = len(all_files)
        final_prefix = f"{target_dir}/{folder_name}".strip("/") if target_dir else folder_name
    
        # Filter file yang belum terupload
        pending_files = []
        for item in all_files:
            dest_hf = f"{final_prefix}/{item['rel_path']}".strip("/")
            if dest_hf in existing_hf_files:
                skip_count += 1
            else:
                item["dest_hf"] = dest_hf
                pending_files.append(item)
            
        print(f"📋 Total: {total_files} file | ⏩ Sudah Ada (Skip): {skip_count} file | ⏳ Perlu di-mirror: {len(pending_files)} file")
    
        if len(pending_files) == 0:
            print("🎉 Semua file sudah ada di Hugging Face! Tidak ada yang perlu di-mirror.")
        else:
            batches = [pending_files[i:i + BATCH_SIZE] for i in range(0, len(pending_files), BATCH_SIZE)]
            total_batches = len(batches)
            print(f"📦 Memproses {len(pending_files)} file dalam {total_batches} Batch (1 Batch = 1 Commit)...\n")
        
            for b_idx, batch in enumerate(batches, 1):
                print("=" * 60)
                print(f"🚀 Memproses BATCH [{b_idx}/{total_batches}] ({len(batch)} file)...")
                temp_batch_dir = tempfile.mkdtemp(prefix=f"batch_{b_idx}_")
                batch_staging = os.path.join(temp_batch_dir, "staging")
                os.makedirs(batch_staging, exist_ok=True)
            
                # 1. Download seluruh file batch ini secara paralel
                print(f"⬇️ Mengunduh {len(batch)} file secara paralel...")
                with ThreadPoolExecutor(max_workers=min(len(batch), 10)) as executor:
                    future_to_item = {}
                    for item in batch:
                        local_f = os.path.join(batch_staging, item["rel_path"])
                        fut = executor.submit(download_file_stream, item["id"], access_token, local_f)
                        future_to_item[fut] = item
                    
                    for fut in as_completed(future_to_item):
                        item = future_to_item[fut]
                        try:
                            fut.result()
                            sz_mb = item["size"] / (1024*1024)
                            print(f"   ✓ Selesai download: {item['name']} ({sz_mb:.1f} MB)")
                        except Exception as e:
                            print(f"   ✕ Gagal download {item['name']}: {e}")
                            fail_count += 1
                        
                # 2. Upload SELURUH BATCH dalam 1 KALI COMMIT (Zero Commit Collision, Anti-429 Rate Limit)
                print(f"⬆️ Mengunggah kloter batch ke Hugging Face: /{final_prefix} (1 Commit)...")
                if upload_folder_batch_safe(batch_staging, final_prefix):
                    print(f"   ✅ BATCH [{b_idx}/{total_batches}] BERHASIL TERUPLOAD (1 Commit)!")
                    ok_count += len(batch)
                else:
                    print(f"   ❌ BATCH [{b_idx}/{total_batches}] GAGAL UPLOAD")
                    fail_count += len(batch)
                
                # 3. Hapus folder sementara kloter ini
                shutil.rmtree(temp_batch_dir, ignore_errors=True)
                print(f"✨ Disk kloter [{b_idx}/{total_batches}] telah dibersihkan.")
                time.sleep(2)
            
    else:
        fname = meta.get("name", "file")
        dest_hf = f"{target_dir}/{fname}".strip("/") if target_dir else fname
        temp_dir = tempfile.mkdtemp(prefix="single_")
        local_f = os.path.join(temp_dir, fname)
        try:
            print(f"⬇️ Mengunduh {fname}...")
            download_file_stream(gdrive_id, access_token, local_f)
            print(f"⬆️ Mengunggah /{dest_hf}...")
            api.upload_file(
                path_or_fileobj=local_f,
                path_in_repo=HF_REPO_ID,
                repo_type="dataset",
                commit_message=f"Mirror: {fname}"
            )
            print(f"✅ Sukses: /{dest_hf}")
            ok_count = 1
        except Exception as e:
            print(f"❌ Gagal: {e}")
            fail_count = 1
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)

    duration = time.time() - start_time
    print("=" * 60)
    print(f"🎉 SELESAI dalam {duration/60:.2f} menit ({duration:.1f} detik)!")
    print(f"✅ Berhasil: {ok_count} file | ⏩ Sudah Ada (Skip): {skip_count} file | ❌ Gagal: {fail_count}")
    print(f"🌐 Website: https://haru-drive.pages.dev")
    print("=" * 60)
